In [3]:
import tensorflow as tf
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
import time
from tqdm import tqdm  # For progress bar

# Check for GPU availability
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

# Load and preprocess the dataset
path_to_file = r"D:\Projects\RNN exp\eng-french\sec\eng_-french.csv"
df = pd.read_csv(path_to_file)
eng_sentences = df['English words/sentences'].values
fr_sentences = ['<start> ' + sent + ' <end>' for sent in df['French words/sentences'].values]

# Tokenize the sentences
eng_tokenizer = tf.keras.preprocessing.text.Tokenizer(filters='')
eng_tokenizer.fit_on_texts(eng_sentences)
eng_sequences = eng_tokenizer.texts_to_sequences(eng_sentences)

fr_tokenizer = tf.keras.preprocessing.text.Tokenizer(filters='')
fr_tokenizer.fit_on_texts(fr_sentences)
fr_sequences = fr_tokenizer.texts_to_sequences(fr_sentences)

# Pad sequences
max_eng_len = 20
max_fr_len = 20
eng_padded = tf.keras.preprocessing.sequence.pad_sequences(eng_sequences, maxlen=max_eng_len, padding='post')
fr_padded = tf.keras.preprocessing.sequence.pad_sequences(fr_sequences, maxlen=max_fr_len, padding='post')

# Split into training and validation sets
eng_train, eng_val, fr_train, fr_val = train_test_split(eng_padded, fr_padded, test_size=0.2)

# Create tf.data.Dataset
BATCH_SIZE = 128
train_dataset = tf.data.Dataset.from_tensor_slices((eng_train, fr_train)).shuffle(len(eng_train)).batch(BATCH_SIZE, drop_remainder=True)
val_dataset = tf.data.Dataset.from_tensor_slices((eng_val, fr_val)).batch(BATCH_SIZE, drop_remainder=True)

# Define Bahdanau Attention
class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = tf.keras.layers.Dense(units)
        self.W2 = tf.keras.layers.Dense(units)
        self.V = tf.keras.layers.Dense(1)

    def call(self, query, values):
        query_with_time_axis = tf.expand_dims(query, 1)
        score = self.V(tf.nn.tanh(self.W1(values) + self.W2(query_with_time_axis)))
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * values
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector, attention_weights

# Define Encoder
class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, enc_units, batch_sz):
        super(Encoder, self).__init__()
        self.batch_sz = batch_sz
        self.enc_units = enc_units
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.lstm = tf.keras.layers.LSTM(self.enc_units, return_sequences=True, return_state=True, recurrent_initializer='glorot_uniform')

    def call(self, x, hidden):
        x = self.embedding(x)
        output, state_h, state_c = self.lstm(x, initial_state=hidden)
        return output, [state_h, state_c]

    def initialize_hidden_state(self):
        return [tf.zeros((self.batch_sz, self.enc_units)), tf.zeros((self.batch_sz, self.enc_units))]

# Define Decoder
class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, dec_units, batch_sz):
        super(Decoder, self).__init__()
        self.batch_sz = batch_sz
        self.dec_units = dec_units
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.lstm = tf.keras.layers.LSTM(self.dec_units, return_sequences=True, return_state=True, recurrent_initializer='glorot_uniform')
        self.fc = tf.keras.layers.Dense(vocab_size)
        self.attention = BahdanauAttention(self.dec_units)

    def call(self, x, hidden, enc_output):
        context_vector, attention_weights = self.attention(hidden[0], enc_output)
        x = self.embedding(x)
        x = tf.concat([tf.expand_dims(context_vector, 1), x], axis=-1)
        output, state_h, state_c = self.lstm(x, initial_state=hidden)
        output = tf.reshape(output, (-1, output.shape[2]))
        x = self.fc(output)
        return x, [state_h, state_c], attention_weights

# Model parameters
embedding_dim = 256
units = 512
eng_vocab_size = len(eng_tokenizer.word_index) + 1
fr_vocab_size = len(fr_tokenizer.word_index) + 1
encoder = Encoder(eng_vocab_size, embedding_dim, units, BATCH_SIZE)
decoder = Decoder(fr_vocab_size, embedding_dim, units, BATCH_SIZE)

# Optimizer and loss function
optimizer = tf.keras.optimizers.Adam()
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# Training step
@tf.function
def train_step(inp, targ, enc_hidden):
    total_loss = 0.0
    total_correct = 0.0
    total_tokens = 0.0
    with tf.GradientTape() as tape:
        enc_output, enc_hidden = encoder(inp, enc_hidden)
        dec_hidden = enc_hidden
        dec_input = tf.expand_dims([fr_tokenizer.word_index['<start>']] * BATCH_SIZE, 1)
        for t in range(1, targ.shape[1]):
            predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_output)
            # Compute loss
            mask = tf.math.logical_not(tf.math.equal(targ[:, t], 0))
            loss_t = loss_object(targ[:, t], predictions)
            masked_loss_t = loss_t * tf.cast(mask, dtype=loss_t.dtype)
            total_loss += tf.reduce_sum(masked_loss_t)
            # Compute accuracy
            predicted_id = tf.argmax(predictions, axis=1)
            correct_t = tf.cast(predicted_id == tf.cast(targ[:, t], predicted_id.dtype), tf.float32) * tf.cast(mask, tf.float32)
            total_correct += tf.reduce_sum(correct_t)
            total_tokens += tf.reduce_sum(tf.cast(mask, tf.float32))
            dec_input = tf.expand_dims(targ[:, t], 1)
        # Average loss for gradient update
        avg_loss = total_loss / total_tokens
    variables = encoder.trainable_variables + decoder.trainable_variables
    gradients = tape.gradient(avg_loss, variables)
    optimizer.apply_gradients(zip(gradients, variables))
    return avg_loss, total_loss, total_correct, total_tokens

# Validation step
@tf.function
def validate_step(inp, targ):
    total_loss = 0.0
    total_correct = 0.0
    total_tokens = 0.0
    enc_hidden = encoder.initialize_hidden_state()
    enc_output, enc_hidden = encoder(inp, enc_hidden)
    dec_hidden = enc_hidden
    dec_input = tf.expand_dims([fr_tokenizer.word_index['<start>']] * BATCH_SIZE, 1)
    for t in range(1, targ.shape[1]):
        predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_output)
        mask = tf.math.logical_not(tf.math.equal(targ[:, t], 0))
        loss_t = loss_object(targ[:, t], predictions)
        masked_loss_t = loss_t * tf.cast(mask, dtype=loss_t.dtype)
        total_loss += tf.reduce_sum(masked_loss_t)
        predicted_id = tf.argmax(predictions, axis=1)
        correct_t = tf.cast(predicted_id == tf.cast(targ[:, t], predicted_id.dtype), tf.float32) * tf.cast(mask, tf.float32)
        total_correct += tf.reduce_sum(correct_t)
        total_tokens += tf.reduce_sum(tf.cast(mask, tf.float32))
        dec_input = tf.expand_dims(targ[:, t], 1)
    return total_loss, total_correct, total_tokens

# Validation function
def validate():
    total_loss = 0.0
    total_correct = 0.0
    total_tokens = 0.0
    for (inp, targ) in val_dataset:
        batch_total_loss, batch_correct, batch_tokens = validate_step(inp, targ)
        total_loss += batch_total_loss
        total_correct += batch_correct
        total_tokens += batch_tokens
    avg_loss = total_loss / total_tokens if total_tokens > 0 else 0
    avg_accuracy = total_correct / total_tokens if total_tokens > 0 else 0
    return avg_loss, avg_accuracy

Num GPUs Available:  0


In [5]:

# Training loop with progress bar and early stopping
EPOCHS = 20
patience = 3
best_val_loss = float('inf')
wait = 0

In [ ]:

for epoch in range(EPOCHS):
    start = time.time()
    enc_hidden = encoder.initialize_hidden_state()
    total_train_loss = 0.0
    total_train_correct = 0.0
    total_train_tokens = 0.0

    # Progress bar with tqdm
    pbar = tqdm(enumerate(train_dataset), total=len(train_dataset), desc=f'Epoch {epoch+1}/{EPOCHS}')
    for batch, (inp, targ) in pbar:
        batch_avg_loss, batch_total_loss, batch_correct, batch_tokens = train_step(inp, targ, enc_hidden)
        total_train_loss += batch_total_loss
        total_train_correct += batch_correct
        total_train_tokens += batch_tokens
        # Update progress bar with running metrics
        running_avg_loss = total_train_loss / total_train_tokens
        running_avg_accuracy = total_train_correct / total_train_tokens
        pbar.set_postfix({'loss': running_avg_loss.numpy(), 'accuracy': running_avg_accuracy.numpy()})

    # Final metrics for the epoch
    avg_train_loss = total_train_loss / total_train_tokens
    avg_train_accuracy = total_train_correct / total_train_tokens
    val_loss, val_accuracy = validate()

    # Print summary in the desired format
    elapsed_time = time.time() - start
    time_per_step = (elapsed_time / len(train_dataset)) * 1000  # ms/step
    print(f'\nEpoch {epoch+1}/{EPOCHS}')
    print(f'{len(train_dataset)}/{len(train_dataset)} ━━━━━━━━━━━━━━━━━━━━ {elapsed_time:.0f}s {time_per_step:.0f}ms/step - accuracy: {avg_train_accuracy:.4f} - loss: {avg_train_loss:.4f} - val_accuracy: {val_accuracy:.4f} - val_loss: {val_loss:.4f}\n')

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        wait = 0
        encoder.save_weights('encoder_weights.weights.h5')
        decoder.save_weights('decoder_weights.weights.h5')
    else:
        wait += 1
        if wait >= patience:
            print("Early stopping triggered!")
            break

print("Training completed with best model weights loaded.")

In [10]:
import tensorflow as tf
import pandas as pd
import numpy as np
from nltk.translate.bleu_score import sentence_bleu

# Load the dataset to rebuild tokenizers
path_to_file = r"D:\Projects\RNN exp\eng-french\sec\eng_-french.csv"
df = pd.read_csv(path_to_file)
eng_sentences = df['English words/sentences'].values
fr_sentences = ['<start> ' + sent + ' <end>' for sent in df['French words/sentences'].values]

# Tokenize the sentences
eng_tokenizer = tf.keras.preprocessing.text.Tokenizer(filters='')
eng_tokenizer.fit_on_texts(eng_sentences)
fr_tokenizer = tf.keras.preprocessing.text.Tokenizer(filters='')
fr_tokenizer.fit_on_texts(fr_sentences)

# Model parameters (must match training)
embedding_dim = 256
units = 512
eng_vocab_size = len(eng_tokenizer.word_index) + 1
fr_vocab_size = len(fr_tokenizer.word_index) + 1
max_eng_len = 20
max_fr_len = 20
BATCH_SIZE = 128

# Define Bahdanau Attention
class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = tf.keras.layers.Dense(units)
        self.W2 = tf.keras.layers.Dense(units)
        self.V = tf.keras.layers.Dense(1)

    def call(self, query, values):
        query_with_time_axis = tf.expand_dims(query, 1)
        score = self.V(tf.nn.tanh(self.W1(values) + self.W2(query_with_time_axis)))
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * values
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector, attention_weights

# Define Encoder
class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, enc_units, batch_sz):
        super(Encoder, self).__init__()
        self.batch_sz = batch_sz
        self.enc_units = enc_units
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.lstm = tf.keras.layers.LSTM(self.enc_units, return_sequences=True, return_state=True, recurrent_initializer='glorot_uniform')

    def call(self, x, hidden):
        x = self.embedding(x)
        output, state_h, state_c = self.lstm(x, initial_state=hidden)
        return output, [state_h, state_c]

    def initialize_hidden_state(self):
        return [tf.zeros((self.batch_sz, self.enc_units)), tf.zeros((self.batch_sz, self.enc_units))]

# Define Decoder
class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, dec_units, batch_sz):
        super(Decoder, self).__init__()
        self.batch_sz = batch_sz
        self.dec_units = dec_units
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.lstm = tf.keras.layers.LSTM(self.dec_units, return_sequences=True, return_state=True, recurrent_initializer='glorot_uniform')
        self.fc = tf.keras.layers.Dense(vocab_size)
        self.attention = BahdanauAttention(self.dec_units)

    def call(self, x, hidden, enc_output):
        context_vector, attention_weights = self.attention(hidden[0], enc_output)
        x = self.embedding(x)
        x = tf.concat([tf.expand_dims(context_vector, 1), x], axis=-1)
        output, state_h, state_c = self.lstm(x, initial_state=hidden)
        output = tf.reshape(output, (-1, output.shape[2]))
        x = self.fc(output)
        return x, [state_h, state_c], attention_weights

# Instantiate models
encoder = Encoder(eng_vocab_size, embedding_dim, units, BATCH_SIZE)
decoder = Decoder(fr_vocab_size, embedding_dim, units, BATCH_SIZE)

# Build models by calling them on dummy data
dummy_input = tf.zeros((BATCH_SIZE, max_eng_len), dtype=tf.int32)
dummy_hidden = encoder.initialize_hidden_state()
enc_output, enc_hidden = encoder(dummy_input, dummy_hidden)
dummy_dec_input = tf.zeros((BATCH_SIZE, 1), dtype=tf.int32)
_ = decoder(dummy_dec_input, enc_hidden, enc_output)

# Load the saved weights
encoder.load_weights('encoder_weights.weights.h5')
decoder.load_weights('decoder_weights.weights.h5')
print("Weights loaded successfully.")

# Translation function
def translate_sentences(sentences, encoder, decoder, eng_tokenizer, fr_tokenizer, max_eng_len, max_fr_len):
    translations = []
    for sentence in sentences:
        # Preprocess input sentence
        input_seq = eng_tokenizer.texts_to_sequences([sentence])
        input_padded = tf.keras.preprocessing.sequence.pad_sequences(input_seq, maxlen=max_eng_len, padding='post')
        input_tensor = tf.convert_to_tensor(input_padded, dtype=tf.int32)

        # Initialize encoder hidden state for single sentence
        batch_size = 1
        enc_hidden = [tf.zeros((batch_size, units)), tf.zeros((batch_size, units))]

        # Encode the input
        enc_output, enc_hidden = encoder(input_tensor, enc_hidden)

        # Initialize decoder input and hidden state
        dec_input = tf.expand_dims([fr_tokenizer.word_index['<start>']], 0)
        dec_hidden = enc_hidden
        result = []

        # Decode step by step
        for t in range(max_fr_len):
            predictions, dec_hidden, _ = decoder(dec_input, dec_hidden, enc_output)
            predicted_id = tf.argmax(predictions[0]).numpy()
            if predicted_id == fr_tokenizer.word_index.get('<end>', 0):
                break
            if predicted_id != 0:  # Skip padding token
                result.append(fr_tokenizer.index_word.get(predicted_id, '<unk>'))
            dec_input = tf.expand_dims([predicted_id], 0)

        translation = ' '.join(result)
        translations.append((sentence, translation))
    return translations

# Test sentences
test_sentences = [
    "Hello",
    "How are you",
    "I love to learn",
    "This is a test",
]

# Translate and print results
translations = translate_sentences(test_sentences, encoder, decoder, eng_tokenizer, fr_tokenizer, max_eng_len, max_fr_len)
for eng, fr in translations:
    print(f"Input: {eng}")
    print(f"Translation: {fr}\n")

Weights loaded successfully.
Input: Hello
Translation: stop !

Input: How are you
Translation: comment allez-vous ?

Input: I love to learn
Translation: j'adore apprendre ça.

Input: This is a test
Translation: c'est un examen.



In [11]:
def evaluate_bleu(reference_translations, predicted_translations):
    bleu_scores = []
    for ref, pred in zip(reference_translations, predicted_translations):
        ref_tokens = [ref.split()]
        pred_tokens = pred.split()
        score = sentence_bleu(ref_tokens, pred_tokens)
        bleu_scores.append(score)
    return np.mean(bleu_scores)

# Example reference translations (replace with actual references)
reference_translations = [
    "Bonjour",
    "Comment vas-tu",
    "J'aime apprendre",
    "Ceci est un test",
]

# Compute BLEU score
predicted_translations = [fr for _, fr in translations]
bleu_score = evaluate_bleu(reference_translations, predicted_translations)
print(f"Average BLEU Score: {bleu_score:.4f}")

Average BLEU Score: 0.0000


D:\Projects\RNN exp\.venv\Lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
D:\Projects\RNN exp\.venv\Lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
D:\Projects\RNN exp\.venv\Lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  war